# Bronze ingest stream

Purpose: Ingest raw HSL vehicle-position messages from Azure Event Hubs Kafka endpoint into a single append-only Bronze raw Delta table.

In [0]:
from pyspark.sql import functions as F

# Catalog / schema / paths
CATALOG_NAME = "hant-catalog"
SCHEMA_NAME = "hsl"

RAW_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.bronze_vehicle_positions_raw"

RAW_PATH = "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/bronze/bronze_raw/hsl_vehicle_positions_raw"
CHECKPOINT_RAW_PATH = "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/bronze/checkpoints/bronze_raw_hsl_vehicle_positions"

# Event Hubs Kafka config
EVENTHUB_NAMESPACE = "hant-event-hub"
EVENTHUB_NAME = "vehicle-position-events"
BOOTSTRAP_SERVERS = f"{EVENTHUB_NAMESPACE}.servicebus.windows.net:9093"

# Databricks secrets
SECRET_SCOPE = "eventhub-scope"
SAS_KEY_NAME_SECRET = "sas-key-name"
SAS_KEY_SECRET = "sas-key"

SAS_KEY_NAME = dbutils.secrets.get(scope=SECRET_SCOPE, key=SAS_KEY_NAME_SECRET)
SAS_KEY = dbutils.secrets.get(scope=SECRET_SCOPE, key=SAS_KEY_SECRET)

KAFKA_SASL_JAAS = (
    'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    'username="$ConnectionString" '
    f'password="Endpoint=sb://{EVENTHUB_NAMESPACE}.servicebus.windows.net/;'
    f'SharedAccessKeyName={SAS_KEY_NAME};'
    f'SharedAccessKey={SAS_KEY};'
    f'EntityPath={EVENTHUB_NAME}";'
)

# Runtime controls
STARTING_OFFSETS = "latest" 
FAIL_ON_DATA_LOSS = "false"
MAX_OFFSETS_PER_TRIGGER = 5000
TRIGGER_INTERVAL = "10 seconds"

# Optional reset flags
RESET_RAW_TABLE = False
RESET_RAW_CHECKPOINT = False

## Object setup

This cell creates the schema and raw Bronze table if missing.

The reset flags are used when intentionally rebuilding the raw stream sink from scratch.

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.{SCHEMA_NAME}")

if RESET_RAW_TABLE:
    for q in spark.streams.active:
        if q.name == "bronze_vehicle_positions_raw_ingest":
            q.stop()

    spark.sql(f"DROP TABLE IF EXISTS {RAW_TABLE}")
    dbutils.fs.rm(RAW_PATH, True)

if RESET_RAW_CHECKPOINT:
    dbutils.fs.rm(CHECKPOINT_RAW_PATH, True)

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {RAW_TABLE} (
    topic                STRING,
    partition            INT,
    offset               BIGINT,
    eventhub_enqueued_ts TIMESTAMP,
    message_key          STRING,
    raw_json             STRING,
    bronze_ingest_ts     TIMESTAMP,
    ingest_date          DATE
)
USING DELTA
PARTITIONED BY (ingest_date)
LOCATION "{RAW_PATH}"
''')

display(spark.sql(f"DESCRIBE DETAIL {RAW_TABLE}"))

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,46ab73f7-43c0-4611-aec5-33d84df68cf2,hant-catalog.hsl.bronze_vehicle_positions_raw,null,abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/bronze/bronze_raw/hsl_vehicle_positions_raw,2026-04-05T18:01:07.47Z,2026-04-16T08:09:34Z,List(ingest_date),List(),389,435785667,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


## Read Event Hubs as Kafka stream

In [0]:
raw_kafka_df = (
    spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", BOOTSTRAP_SERVERS)
        .option("subscribe", EVENTHUB_NAME)
        .option("kafka.security.protocol", "SASL_SSL")
        .option("kafka.sasl.mechanism", "PLAIN")
        .option("kafka.sasl.jaas.config", KAFKA_SASL_JAAS)
        .option("startingOffsets", STARTING_OFFSETS)
        .option("failOnDataLoss", FAIL_ON_DATA_LOSS)
        .option("maxOffsetsPerTrigger", MAX_OFFSETS_PER_TRIGGER)
        .load()
)

## Minimal raw Bronze projection

In [0]:
bronze_raw_stream = (
    raw_kafka_df
        .select(
            F.col("topic").cast("string").alias("topic"),
            F.col("partition").cast("int").alias("partition"),
            F.col("offset").cast("bigint").alias("offset"),
            F.col("timestamp").alias("eventhub_enqueued_ts"),
            F.col("key").cast("string").alias("message_key"),
            F.col("value").cast("string").alias("raw_json"),
            F.current_timestamp().alias("bronze_ingest_ts")
        )
        .withColumn("ingest_date", F.to_date("bronze_ingest_ts"))
)

## Start raw Bronze ingest query

In [0]:
# Stop an older query with the same name before restarting
for q in spark.streams.active:
    if q.name == "bronze_vehicle_positions_raw_ingest":
        q.stop()

bronze_raw_query = (
    bronze_raw_stream.writeStream
        .queryName("bronze_vehicle_positions_raw_ingest")
        .format("delta")
        .outputMode("append")
        .trigger(processingTime=TRIGGER_INTERVAL)
        .option("checkpointLocation", CHECKPOINT_RAW_PATH)
        .toTable(RAW_TABLE)
)

print("Raw Bronze stream started.")
print("  Query name :", bronze_raw_query.name)
print("  Query ID   :", bronze_raw_query.id)
print("  Status     :", bronze_raw_query.status)

Raw Bronze stream started.
  Query name : bronze_vehicle_positions_raw_ingest
  Query ID   : 84e83ef4-9af3-411c-8956-a35cce86531f
  Status     : {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


## Lightweight monitoring

In [0]:
for q in spark.streams.active:
    if q.name == "bronze_vehicle_positions_raw_ingest":
        print("NAME:", q.name)
        print("ID:", q.id)
        print("IS ACTIVE:", q.isActive)
        print("STATUS:", q.status)
        print("LAST PROGRESS:", q.lastProgress)
        print("EXCEPTION:", q.exception())
        break

NAME: bronze_vehicle_positions_raw_ingest
ID: 84e83ef4-9af3-411c-8956-a35cce86531f
IS ACTIVE: True
STATUS: {'message': 'Getting offsets from KafkaV2[Subscribe[vehicle-position-events]]', 'isDataAvailable': False, 'isTriggerActive': True}
LAST PROGRESS: None
EXCEPTION: None
